# Exercise 13 : Data Formats and Tools

Count the 2024 road deaths per road-user category, driver (`DRIV`), passenger (`PAS`), pedestrian
(`PED`) and unknown (`UNK`), and the number of countries that report each category, with pandas,
DuckDB and Polars. Use the same selection in every tool and sort the rows by category code.

Read `datasets/eurostat-road-safety-README.md` first. `eurostat-road-deaths-2023-2024.csv` is an
unchanged Eurostat extract of 12,254 rows, retrieved on 10 October 2026. Each row holds one value
for a combination of unit, sex, age, road-user category, country and year. Missing data are absent
rows ; no value is empty.

In [ ]:
from pathlib import Path

import duckdb
import pandas as pd
import polars as pl
import pyarrow.parquet as pq

DATA = Path.cwd().parents[1] / 'datasets'

## Step 1 : Typed load

Read the file with `pd.read_csv()`. Read only these columns with `usecols` :

- `LAST UPDATE`, parsed with `parse_dates` and `date_format='%d/%m/%y %H:%M:%S'`
- The codes `sex`, `age`, `unit`, `pers_cat` and `geo`, with the dtype `category`
- `TIME_PERIOD` and `OBS_VALUE`, with the nullable dtype `Int64`
- `OBS_FLAG`, with the dtype `str`

Rename `LAST UPDATE`, `TIME_PERIOD`, `OBS_VALUE` and `OBS_FLAG` to `last_update`, `year`, `value`
and `flag`. Print the dtypes and the first release time : 25 August 2026, 23:00. Check that no
value is missing and that no combination of the five codes and the year occurs twice.

In [ ]:
deaths = pd.read_csv(
    DATA / 'eurostat-road-deaths-2023-2024.csv',
    usecols=['LAST UPDATE', 'sex', 'age', 'unit', 'pers_cat', 'geo',
             'TIME_PERIOD', 'OBS_VALUE', 'OBS_FLAG'],
    dtype={'sex': 'category', 'age': 'category', 'unit': 'category',
           'pers_cat': 'category', 'geo': 'category',
           'TIME_PERIOD': 'Int64', 'OBS_VALUE': 'Int64', 'OBS_FLAG': 'str'},
    parse_dates=['LAST UPDATE'], date_format='%d/%m/%y %H:%M:%S',
).rename(columns={'LAST UPDATE': 'last_update', 'TIME_PERIOD': 'year',
                  'OBS_VALUE': 'value', 'OBS_FLAG': 'flag'})
print(deaths.dtypes)
print(deaths['last_update'].iloc[0])
print(deaths['value'].isna().sum())
print(deaths.duplicated(['sex', 'age', 'unit', 'pers_cat', 'geo', 'year']).sum())

## Step 2 : pandas

Select the rows with :

- `unit == 'NR'`, the counts
- `sex == 'T'` and `age == 'TOTAL'`
- The year 2024
- Countries only, without `EU27_2020`

Leave out the road-user category `TOTAL`. Group by `pers_cat` with `observed=True`, sum the values
as `deaths` and count the rows as `countries`.

Check : 126 rows ; `DRIV` 14,078 deaths in 31 countries, `PAS` 3,007 in 31, `PED` 3,706 in 31 and
`UNK` 84 in 33. Check that the categories add up to the 20,875 deaths of the `TOTAL` rows in the
same selection. Why does `UNK` have 33 countries and the other categories 31?

In [ ]:
in_scope = (deaths['unit'].eq('NR') & deaths['sex'].eq('T') & deaths['age'].eq('TOTAL')
            & deaths['year'].eq(2024) & ~deaths['geo'].isin(['EU27_2020']))
total = deaths.loc[in_scope & deaths['pers_cat'].eq('TOTAL'), 'value'].sum()
selected = deaths.loc[in_scope & deaths['pers_cat'].ne('TOTAL')]

answer = (selected.groupby('pers_cat', observed=True)
          .agg(deaths=('value', 'sum'), countries=('value', 'size'))
          .reset_index())
print(len(selected))
print(answer)
print(answer['deaths'].sum(), total)

## Step 3 : Parquet

Write the loaded DataFrame to `deaths.parquet` with `index=False` and read it back. Compare the
dtypes before and after. Print the stored schema with `pyarrow.parquet.read_schema()` : how does
Parquet store the categories? Read only the columns `pers_cat`, `geo`, `year` and `value`.

In [ ]:
deaths.to_parquet('deaths.parquet', index=False)
restored = pd.read_parquet('deaths.parquet')
print(restored.dtypes.equals(deaths.dtypes))
print(pq.read_schema('deaths.parquet'))

subset = pd.read_parquet('deaths.parquet', columns=['pers_cat', 'geo', 'year', 'value'])
print(subset.shape)

## Step 4 : DuckDB

Repeat the count with DuckDB : `con = duckdb.connect()` and a query
`FROM read_parquet('deaths.parquet')` with the same filters, `GROUP BY pers_cat` and
`ORDER BY pers_cat`. Name the sum `deaths` and the count `countries`, and fetch the result with
`.df()`. Are the deaths and countries the same as in pandas?

In [ ]:
con = duckdb.connect()
from_sql = con.execute("""
    SELECT pers_cat, CAST(SUM(value) AS INTEGER) AS deaths, COUNT(*) AS countries
    FROM read_parquet('deaths.parquet')
    WHERE unit = 'NR' AND sex = 'T' AND age = 'TOTAL' AND year = 2024
      AND geo <> 'EU27_2020' AND pers_cat <> 'TOTAL'
    GROUP BY pers_cat ORDER BY pers_cat
""").df()
print(from_sql)
print(from_sql['deaths'].tolist() == answer['deaths'].tolist(),
      from_sql['countries'].tolist() == answer['countries'].tolist())

## Step 5 : Polars

Repeat it with a lazy query : `pl.scan_parquet('deaths.parquet')`, expressions in `.filter()`,
`.group_by('pers_cat')`, `.agg()` with `pl.col('value').sum()` and `pl.len()`, and
`.sort('pers_cat')`. Print `.explain()` and find the Parquet scan, the selected columns and the
filter. Run the query with `.collect()` and compare the result with pandas.

In [ ]:
query = (pl.scan_parquet('deaths.parquet')
         .filter((pl.col('unit') == 'NR') & (pl.col('sex') == 'T')
                 & (pl.col('age') == 'TOTAL') & (pl.col('year') == 2024)
                 & (pl.col('geo') != 'EU27_2020') & (pl.col('pers_cat') != 'TOTAL'))
         .group_by('pers_cat')
         .agg(pl.col('value').sum().alias('deaths'), pl.len().alias('countries'))
         .with_columns(pl.col('pers_cat').cast(pl.String))
         .sort('pers_cat'))
print(query.explain())
from_polars = query.collect()
print(from_polars)
print(from_polars['deaths'].to_list() == answer['deaths'].tolist())

## Step 6 : Reflection

Write 5–8 sentences in the answer cell. Explain the selection, why the `TOTAL` rows and
`EU27_2020` are left out, and why an absent row is not a zero. Which tool would you choose for
interactive inspection, and which for repeated queries over a large Parquet file? Mention column
selection and memory, and the difference between Arrow and Parquet.

The selection keeps the counts (`NR`) for both sexes and all ages in 2024, so each death is
counted once. The `TOTAL` rows and `EU27_2020` are sums of other rows : keeping them would count
the same deaths twice. A country without a row for a category did not report it, so its value is
unknown. pandas suits interactive inspection : every intermediate result is a DataFrame to print.
For repeated queries over a large Parquet file, DuckDB or a lazy Polars query reads only the
columns that the query needs and can skip row groups by their statistics, which saves memory.
Arrow is the columnar format in memory that the three tools share ; Parquet is the compressed
columnar format on disk.

## If time permits :

- Run `demos/13-data-formats-and-tools/demo08-numba-optional.py` after
  `uv sync --frozen --all-groups`. Check that the three results agree before you compare the timings